In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'

sns.set(font_scale=1.2,style="whitegrid")
%matplotlib inline

from tjn_tools.data_processing import *
from tjn_internal.style_guide import *

#Avoid overlapping text|
from adjustText import adjust_text
from datetime import datetime


import statsmodels.formula.api as smf
from itertools import permutations, product
import pickle

pd.set_option('display.max_columns', 100)

from config import CBCR_FILE, ORBIS_FILE, CORPORATE_TAX_RATE_PATH, YEAR_CBCR, UNILATERAL_CROSS, UNILATERAL_PANEL, BILATERAL_CROSS, GRAV_FILE, LINK_FILE

In [ ]:
path_files = "../data/raw/"
path_files_final = "../data/final/"
# now = datetime.now().year
last_year_mean = YEAR_CBCR-6 

## Section 1
#Output
etr_output = f"{path_files_final}{YEAR_CBCR}_cbcr_etr_rates.xlsx"
cbcr_output = f"{path_files_final}{YEAR_CBCR}_cbcr_cleaned_dividends.xlsx"

## Section 2
#Output
df_pairs_output = f"{path_files_final}{YEAR_CBCR}_df_pairs.tsv"
df_fin_output = f"{path_files_final}{YEAR_CBCR}_df_fin.tsv"
d_groups2countries_output = f"{path_files_final}{YEAR_CBCR}_d_groups2countries.dump"
iso3_to_wages_output = f"{path_files_final}{YEAR_CBCR}_iso3_to_wages.dump"
iso3_to_cit_output = f"{path_files_final}{YEAR_CBCR}_iso3_to_cit.dump"
iso3_to_gdp_output = f"{path_files_final}{YEAR_CBCR}_iso3_to_gdp.dump"
info_expenditures_output = f"{path_files_final}{YEAR_CBCR}_info_expenditures.csv"



In [ ]:
#OFCs
ofcs = {'MLT', 'BRB', 'BMU', 'CUW', 'CYP', 'HKG', 'PRI', 'SGP', 'IRL', 'MUS', 'VGB', 'CYM', 
              'LUX', 'JEY', 'IMN', 'NLD', 'GIB', 'CHE', 'BHS', 'GGY', 'MAC', 'HUN', 'ARE'}
groups = {"ASIAT","EUROP","AMER","AFRIC",'OTE','OAM',"OAF","OAS"} 
#Other america and other europe usualyl contain many tax havens. When split only by continent, remove across all conutries


In [ ]:
#CIT rates
cit_file = CORPORATE_TAX_RATE_PATH 
iso3_to_cit  = pd.read_csv(cit_file)
iso3_to_cit = iso3_to_cit.loc[iso3_to_cit["year"]==YEAR_CBCR].groupby("country_iso3")["nctr_final"].mean()

iso3_to_cit["MLT"] *= 1/7 #(6/7th rule)
iso3_to_cit["GIB"] = 0 #only resident income
iso3_to_cit["MCO"] = 0
iso3_to_cit["AND"] = 0
iso3_to_cit["CAF"] = 0.3
iso3_to_cit["HTI"] = 0.3
iso3_to_cit["YEM"] = 0.2
iso3_to_cit["NCL"] = 0 #Only NCL income is taxable
iso3_to_cit["PRK"] = 0.325
iso3_to_cit["COD"] = 0.28
iso3_to_cit["TLS"] = 0.10
#iso3_to_cit["ROU"] = iso3_to_cit["ROM"]
iso3_to_cit["USA"] = 0.27 # State tax included

pickle.dump(iso3_to_cit,open(iso3_to_cit_output,"wb+"))

In [ ]:
for i in set(iso3_to_cit[np.isnan(iso3_to_cit)].index):
    print(i,iso3_to_name(i))

# 1. Clean CBCR data and calculate ETR
- Input file: CBCR (https://stats.oecd.org/Index.aspx?DataSetCode=CBCR_TABLEI)
- Orbis with number of expected companies
    - Columns needed:
        - Country ISO code
        - Operating revenue
        - Number of employees
        - GUO code
    - Search strategy:
        - Status: Active companies, Unknown situation
        - Operating revenue (Turnover), using estimates (m USD): min=850, Last available year, exclusion of companies with no recent financial data and Public authorities/States/Governments
        - Ultimate Owners: Global; Def. of the UO: min. path of 50.01%, known or unknown shareholder
        - Shareholders with foreign subsidiaries: located anywhere (excluding unknown countries) not ultimately owned but at least 51% owned; May have other shareholder in the foreign country; Def. of the UO: min. path of 50.01%, known or unknown shareholder
        - NACE Rev. 2, core code (4 digits): Different from 65


In [ ]:
fin_needed = ['Unrelated Party Revenues','Profit (Loss) before Income Tax', 'Adjusted Profit (Loss) before Income Tax',
       'Income Tax Paid (on Cash Basis)','Income Tax Accrued - Current Year','Number of Employees',
       'Tangible Assets other than Cash and Cash Equivalents','Number of CbCRs','Number of CbCR Sub Groups','Number of Entities']

In [ ]:
## Functions to calculate ETRs
def calculate_etr(df):
    d = df.loc[df["Income Tax Paid (on Cash Basis)"]>=0].groupby("JUR").sum()
    d["etr"] = d["Income Tax Paid (on Cash Basis)"]/d["Profit (Loss) before Income Tax"]
    return d["etr"]

def main_etrs(year,file,g="Sub-Groups with positive profits"):
    if isinstance(file,str):
        df = pd.read_csv(file,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
        df = df.loc[df["Grouping"]==g]
        df = df.loc[df["Variable"].isin(fin_needed)]
        df = df.loc[df["Year"]==year]
        df = pd.pivot_table(df,index=["COU","JUR","Partner Jurisdiction"],values="Value",columns="Variable").reset_index()
    else:
        df = file
    df = df.dropna(subset=["Income Tax Paid (on Cash Basis)"])

    df_for = df.loc[df["COU"] != df["JUR"]]
    df_dom = df.loc[df["COU"] == df["JUR"]]
    df_tot = df

    df_etr = pd.concat([calculate_etr(df_for),
                       calculate_etr(df_dom),
                       calculate_etr(df_tot)],
                      axis=1)
    df_etr.columns = ["ETR_foreign","ETR_domestic","ETR_total"]
    return df_etr
#     df_etr.to_excel(etr_output)

#     df_etr.head()

## 1.1 CBCR vs Orbis: Compare coverage

In [ ]:
#CBCR number of companies
total_cov = pd.read_csv(CBCR_FILE,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
total_cov = total_cov.loc[total_cov["Year"]==YEAR_CBCR]
total_cov = total_cov.loc[total_cov["Grouping"]=="All Sub-Groups"]
total_cov = total_cov.loc[total_cov["Variable"]=="Number of CbCRs"]
total_cov = total_cov.loc[total_cov["JUR"]==total_cov["COU"]]
total_cov = total_cov.groupby(["COU"]).max()["Value"]

## Compare quality with Orbis (companies with >750M of turnover, including estimates, and keeping companies with foreign subsidiaries)
cs = pd.read_excel(ORBIS_FILE,na_values=["n.a."],keep_default_na=False)
cs = cs.dropna(subset=["GUO - BvD ID number"])
cs = cs.loc[(cs["Country ISO code"]==cs["GUO - BvD ID number"].str[:2])|(cs["GUO - BvD ID number"].str[:2]=="WW")]
cs["max_turnover"] = cs.groupby(["GUO - BvD ID number"])["Operating revenue (Turnover)\nth USD Last avail. yr"].transform(max)
cs = cs.loc[cs["Operating revenue (Turnover)\nth USD Last avail. yr"]==cs["max_turnover"]]
print(cs["GUO - BvD ID number"].value_counts().head()) #makek sure no repeated, in case of equal turnovers

cs = cs.drop_duplicates(subset=["GUO - BvD ID number"])
cs.loc[(cs["Country ISO code"]!=cs["GUO - BvD ID number"].str[:2])&(cs["GUO - BvD ID number"].str[:2]!="WW"),"Country ISO code"] = cs.loc[(cs["Country ISO code"]!=cs["GUO - BvD ID number"].str[:2])&(cs["GUO - BvD ID number"].str[:2]!="WW"),"GUO - BvD ID number"].str[:2]
print(cs["GUO - BvD ID number"].value_counts().head()) #makek sure no repeated, in case of equal turnovers

cs = cs.groupby("Country ISO code").agg({"GUO - BvD ID number": len,"Operating revenue (Turnover)\nth USD Last avail. yr":np.sum,"Number of employees\nLast avail. yr":np.sum}).reset_index()
cs.columns = ["country","Number companies expected","Turnover expected","Emp expected"]
cs["Turnover expected"] *= 1000
cs["country"] = cs["country"].map(get_iso3)
cs = cs.dropna(subset=["country"])

#This array is used in 99.cluster_photoshop
potential_countries = cs.loc[cs["Number companies expected"]>=3,"country"].values

cs = pd.concat([cs.set_index("country"),total_cov],axis=1)
cs["Coverage"] = np.log2(cs["Value"]/cs["Number companies expected"])


cs_plot = cs.dropna()
plt.figure(figsize=(8,6))
plt.plot(cs_plot["Number companies expected"],cs_plot["Coverage"],"o",color="gray")

texts = []
for c,x,y in zip(cs_plot.index,cs_plot["Number companies expected"],cs_plot["Coverage"]):
    texts.append(plt.text(x,y,iso3_to_name(c)))


plt.plot([4,1800],[0,0],alpha=0.5,color="gray")             
plt.xscale("log")

adjust_text(texts,cs_plot["Number companies expected"].values, cs_plot["Coverage"].values, 
            arrowprops=dict(arrowstyle='->', color='gray'))
bins = np.array([0.25,0.5,0.75,1,1.5,2,4])
coverage = np.log2(bins)
plt.yticks(coverage,np.array(100*bins,dtype=int))
plt.xlabel("Expected firms")
plt.ylabel("Coverage (%)")
sns.despine(bottom=False,left=True)


expected_companies = cs["Number companies expected"]
expected_companies = expected_companies.to_dict()
expected_companies["DNK"]

In [ ]:
cs_plot["Ratio"] = cs_plot["Number companies expected"]/cs_plot["Value"]
x = cs_plot[["Number companies expected","Value","Ratio"]].sort_values(by="Ratio", ascending=False)
x.index = x.index.map(iso3_to_name)
x["Number companies expected"] = x["Number companies expected"].astype(int)
x["Value"] = x["Value"].astype(int)
x["Ratio"] = x["Ratio"].round(2)
x

In [ ]:
potential_countries

In [ ]:
df = pd.read_csv(CBCR_FILE,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
# df = df.loc[df["Grouping"]=="Sub-Groups with positive profits"]
df = df.loc[df["Grouping"]=="All Sub-Groups"]
df = df.loc[df["Year"]==YEAR_CBCR]
df = df.loc[df["Variable"].isin(fin_needed)]
df = pd.pivot_table(df,index=["COU","JUR","Partner Jurisdiction"],values="Value",columns="Variable").reset_index()
if "Adjusted Profit (Loss) before Income Tax" in list(df.columns):
    df.dropna(subset=["Adjusted Profit (Loss) before Income Tax"])[["JUR","Profit (Loss) before Income Tax", "Adjusted Profit (Loss) before Income Tax"]].set_index("JUR")/1E9

In [ ]:
df = pd.read_csv(CBCR_FILE)
df = df.loc[df["Year"]==2017]
df1 = df.loc[df["Grouping"]=="Sub-Groups with positive profits"]
a = df1[["COU","Partner Jurisdiction"]].drop_duplicates().reset_index(drop=True)["COU"].value_counts()

df1 = df.loc[df["Grouping"]=="All Sub-Groups"]
b = df1[["COU","Partner Jurisdiction"]].drop_duplicates().reset_index(drop=True)["COU"].value_counts()

pd.concat([b,a],axis=1).transpose().fillna(0).astype(int)

In [ ]:
#Make sure the cleaning in 2016  (multiplying everything by 2 in china) was mostly okay
df = pd.read_csv(CBCR_FILE,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
df = df.loc[df["Grouping"]=="Sub-Groups with positive profits"]
# df = df.loc[df["Grouping"]=="All Sub-Groups"]
df = df.loc[df["Variable"].isin(fin_needed+['Number of CbCRs'])]
df = pd.pivot_table(df,index=["Year","COU","JUR","Partner Jurisdiction"],values="Value",columns="Variable").reset_index()
# df.dropna(subset=["Adjusted Profit (Loss) before Income Tax"])[["JUR","Profit (Loss) before Income Tax", "Adjusted Profit (Loss) before Income Tax"]].set_index("JUR")/1E9
df.loc[(df["JUR"]=="COD")]


## 1.2 Read and clean CBCR
- Data limitations: https://www.oecd.org/tax/tax-policy/anonymised-and-aggregated-cbcr-statistics-disclaimer.pdf
- Report: https://www.oecd.org/tax/tax-policy/corporate-tax-statistics-database.htm


- Data on the United States --> Cleaned in the paper with Zucman
    - US domestic: 55%
    - US foreign: 7% (https://gabriel-zucman.eu/files/GBJZ2021.pdf)

- Other countries: correct using notes when possible:
    - Ireland: https://www.oecd.org/tax/tax-policy/ireland-cbcr-country-specific-analysis.pdf
        --> No issues found
    - Italy: https://www.oecd.org/tax/tax-policy/italy-cbcr-country-specific-analysis.pdf
        -->  The average value of the share of dividends is instead equal to 38.2%, thus implying that dividends are concentrated in few firms.  
    - Netherlands: https://www.oecd.org/tax/tax-policy/united-kingdom-cbcr-country-specific-analysis.pdf
        --> 5794 out of 36802 = 15.74%
    - United Kingdom: https://www.oecd.org/tax/tax-policy/united-kingdom-cbcr-country-specific-analysis.pdf
        --> UK reported profit.: The total value of dividends extracted for groups we believe had included intragroup dividends received was approximately £55 billion. (78.169865 out of 152.918884 = 51.1%) 	)
    - Sweden: https://www.oecd.org/tax/tax-policy/sweden-cbcr-country-specific-analysis.pdf
        --> Dividends share 51.95%


Correct for dividends, but not for participation results (de-mergers, takeovers and disposal typically involve tax avoidance strategies in tax havens.) 


Cleaning strategy based on the analysis below:
- Profit and coordination centers:
    - Domestic: Analysis done for those countries (except Belgium and LUX)
        - Belgium: 50% (same as uk/swe)
        - LUX: Nothing (tax rates are usually around there)
    - Foreign : 
        -Remove 10% (a bit more than what we find in the US for that year (9% = 42 double counting/454 tax havens))
- USA:
    - Domestic: 
        - Remove 35%
    - Foreign: 
        - No correction


- Other countries:
    - Domestic:
        - If domestic etr << foreign etr (10 points): Correct as in ita (35%)
    - Foreign: 
        - No correction

- Groups:
    - Domestic:
        - N/A
    - Foreign:
        - Remove 5% (10% from tax havens, with 50% of profits in tax havens)


In [ ]:
#Check out cleaning. USe all groups to get a most comparable picture
df_etrs = main_etrs(year=YEAR_CBCR,file=CBCR_FILE,g="All Sub-Groups")
df_etrs["cit"] = df_etrs.index.map(iso3_to_cit)
df_etrs

plt.figure(figsize=(12,5))
for i,row in df_etrs.iterrows():
    if row["ETR_domestic"]>-0.2:
        if i in ["NLD","GBR","ITA","SWE"]:
            color = "blue"
        else:
            color = "black"
        plt.annotate(i,(row["ETR_domestic"],row["ETR_foreign"]),color=color)
plt.plot(df_etrs["ETR_domestic"],df_etrs["ETR_foreign"],"o")


def plot_c(cs,color="tomato"):
    c,fraction = cs
    plt.plot(df_etrs.loc[c,"ETR_domestic"]*(1+fraction/(1-fraction)),df_etrs.loc[c,"ETR_foreign"],".",color=color)
    plt.plot([df_etrs.loc[c,"ETR_domestic"]*(1+fraction/(1-fraction)),df_etrs.loc[c,"ETR_domestic"]],[df_etrs.loc[c,"ETR_foreign"],df_etrs.loc[c,"ETR_foreign"]],lw=1,color=color)

d_reduction = dict()
for c in df_etrs.dropna(subset=["ETR_domestic"]).index:
    d = dict([("SWE",0.52),("ITA",0.38),("USA",0.35)]) #2017
#     d = dict([("SWE",0.52),("ITA",0.38)]) #2016
    
    if c in ["BEL","SGP","IMN","BMU"]: 
        plot_c((c,0.5),color="red")
        frac = 0.5
    elif c in ["SVN","MEX"]: #2017
#     elif c in ["SVN","USA","CHN","MEX"]: #2016
        frac = 0
    elif c in d:
        plot_c((c,d[c]),color="cornflowerblue")
        frac = d[c]
    elif c in ("GBR","NLD"): #2017
#     elif c in []: #2016
        frac = 0 #They report their figure of adjusted profits
    elif c == "LUX": #negative profits
        frac = 0
    else:
        frac = 0.35
    d_reduction[c] = frac
    plot_c((c,frac),color="gray")
        
plt.plot([0,0.4],[0,0.4],zorder=0)
plt.grid(axis="y")
sns.despine(left=True,bottom=True)
plt.xlabel("ETR (Domestic MNCs)")
plt.ylabel("ETR (Foreign MNCs)")
# plt.plot([0,0.4],[0.05,0.45])

In [ ]:
# Manually adjust countries without domestic ETRs for "sub-groups with positive profits". Use all sub-groups in this case
df = pd.read_csv(CBCR_FILE,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
df = df.loc[df["Year"]==YEAR_CBCR]
missing_c = (set(df.loc[(df["COU"]==df["JUR"])&(df["Grouping"]=="All Sub-Groups"),"COU"]) -
         set(df.loc[(df["COU"]==df["JUR"])&(df["Grouping"]=="Sub-Groups with positive profits"),"COU"]))
df = df.loc[df["Grouping"]=="All Sub-Groups"]
df = pd.pivot_table(df,index=["Year","COU","JUR","Partner Jurisdiction"],values="Value",columns="Variable").reset_index()
#Calculate etr
display(main_etrs(YEAR_CBCR,df.loc[(df["JUR"].isin(missing_c))]))

#Manually adjust
d_reduction["AUT"] = 0.35 #2017 only, much lower etr domestic
d_reduction["LVA"] = 0 #2017 only, higher for etr domestic
d_reduction["POL"] = 0.35 #2016 and 2017much lower etr domestic

In [ ]:

#Clean data CBCR
df = pd.read_csv(CBCR_FILE,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
df = df.loc[df["Variable"].isin(fin_needed)]
# df = df.loc[df["Grouping"]=="Sub-Groups with positive profits"]
df = df.loc[df["Year"]==YEAR_CBCR]
df = pd.pivot_table(df,index=["Grouping","COU","JUR","Partner Jurisdiction"],values="Value",columns="Variable").reset_index()
df = df.dropna(subset=["Income Tax Paid (on Cash Basis)"])


#Countries with analysis
if "Adjusted Profit (Loss) before Income Tax" in list(df.columns):
    print("Cleaning countries with corrected data: NLD and GBR")
    display(df.loc[~np.isnan(df["Adjusted Profit (Loss) before Income Tax"])])
    df.loc[~np.isnan(df["Adjusted Profit (Loss) before Income Tax"]),"Profit (Loss) before Income Tax"] = df.loc[~np.isnan(df["Adjusted Profit (Loss) before Income Tax"]),"Adjusted Profit (Loss) before Income Tax"]

print("Cleaning countries with analysis: ITA, SWE, BEL, USA")
df["Reduction"] = df["COU"].map(d_reduction).fillna(0)

filter_ = (df["COU"]==df["JUR"])&(df["Profit (Loss) before Income Tax"]>0)

d1 = df.loc[filter_].groupby(["COU","Grouping"])["Profit (Loss) before Income Tax"].sum()/1E9
df.loc[filter_,"Profit (Loss) before Income Tax"] = df.loc[filter_,"Profit (Loss) before Income Tax"]*(1-df.loc[filter_,"Reduction"])
d2 = df.loc[filter_].groupby(["COU","Grouping"])["Profit (Loss) before Income Tax"].sum()/1E9
d = pd.concat([d1,d2],axis=1).reset_index()
display(d.loc[d["Grouping"]=="All Sub-Groups"])

print("Cleaning foreign activities in tax havens")
filter_ = (df["COU"]!=df["JUR"]) & (df["JUR"].isin(ofcs))&(df["Profit (Loss) before Income Tax"]>0)
print(df.loc[filter_,"Profit (Loss) before Income Tax"].sum()/1E9)
df.loc[filter_,"Profit (Loss) before Income Tax"] = df.loc[filter_,"Profit (Loss) before Income Tax"]*(1-0.1)
print(df.loc[filter_,"Profit (Loss) before Income Tax"].sum()/1E9)

print("Cleaning foreign activities in groups")
filter_ = (df["COU"]!=df["JUR"]) & (df["JUR"].isin(groups | {"FJT"}))&(df["Profit (Loss) before Income Tax"]>0)
print(df.loc[filter_,"Profit (Loss) before Income Tax"].sum()/1E9)
df.loc[filter_,"Profit (Loss) before Income Tax"] = df.loc[filter_,"Profit (Loss) before Income Tax"]*(1-0.05)
print(df.loc[filter_,"Profit (Loss) before Income Tax"].sum()/1E9)


df.to_excel(cbcr_output)

In [ ]:
# df = pd.read_csv(CBCR_FILE,usecols=["COU","JUR","Year","Partner Jurisdiction","Grouping","Variable","Value"])
df.loc[(df["COU"]=="CHN")].sort_values(by="Profit (Loss) before Income Tax").tail(10)

In [ ]:
df_etrs = main_etrs(year=YEAR_CBCR,file=df.loc[df["Grouping"]=="Sub-Groups with positive profits"])
df_etrs["cit"] = df_etrs.index.map(iso3_to_cit)
df_etrs.loc[df_etrs["ETR_foreign"]>0.6,"ETR_foreign"] = df_etrs.loc[df_etrs["ETR_foreign"]>0.6,"cit"]
df_etrs.loc[df_etrs["ETR_total"]>0.6,"ETR_total"] = df_etrs.loc[df_etrs["ETR_total"]>0.6,"cit"]
df_etrs.to_excel(etr_output)
df_etrs.loc["AUT"]

In [ ]:
df_etrs2 = main_etrs(year=YEAR_CBCR,file=CBCR_FILE,g="Sub-Groups with positive profits")
d = pd.concat([df_etrs, df_etrs2],axis=1)
d.columns = ["f1","d1","t1","cit","f2","d2","t2"]

def plot_etr_robustness(d,v2):
    plt.plot(d["f1"],d[v2],"o")
    plt.plot([0,0.45],[0,0.45],"--",color="gray",zorder=0)
    plt.xlabel("ETR (foreign MNCs)")
    plt.ylabel("ETR (domestic MNCs)")
    sns.despine(bottom=True,left=True)
    
plt.figure(figsize=(13,4))
plt.subplot(121)
plot_etr_robustness(d,"d2")
plt.title("(A) Original data")

plt.subplot(122)
plot_etr_robustness(d,"d1")
plt.title("(B) Corrected for double counting")

In [ ]:
#Fix code for kosovo
df["COU"] = df["COU"].replace({"XKV": "XKX"})
df["JUR"] = df["JUR"].replace({"XKV": "XKX"})

In [ ]:
# df_fin.loc[df_fin["iso3_o"]=="USA"].sort_values(by="pi",ascending=False)

## 2. Create other variables to train the model

## 2.1. Financial information

In [ ]:
#Financial variables
df_fin = df.loc[df["Grouping"]=="All Sub-Groups",["COU","JUR","Number of Employees","Profit (Loss) before Income Tax","Income Tax Paid (on Cash Basis)","Unrelated Party Revenues","Tangible Assets other than Cash and Cash Equivalents"]]
df_fin.columns = ["iso3_o","iso3_d","emp","pi","txc","revt","t_at"]
df_fin["iso3_d"] = df_fin["iso3_d"].replace("ANT","CUW")


In [ ]:
## Countries to include
countries_include = set([_ for _ in df_fin["iso3_d"].unique() if not isinstance(get_iso3(_),float)]) - {"IOT","BVT"}# - {"BVT","GUM","ASM","VIR"} #tiny ocuntry with almost no profits, not in the distance dataset

## 2.2 Basic info and impute missing wages

In [ ]:

## Read region
region = pd.read_csv(UNILATERAL_CROSS,skiprows=1,sep="\t",usecols=["iso3","region_tjn","region","UKt","OECD","OECD_OCT","EU28","G20",
                                                               "FSI2020_Rank","FSI2020_Share","FSI2020_Score","CTHI21_Rank","CTHI21_Share","CTHI21_Score"])

##Other info
usecols = ["iso3","year","GDP_int","POP_int","Physicians_per_1000_wb","Nurses_per_1000_wb","GreenfieldFDI_inward","GreenfieldFDI_outward",
           "resource_revenue","resource_taxes","total_taxes_revenue","cit_revenue","Health_exp_gdp_wb","Govt_exp_educ_gdp_wb",
          "FDI_inflows_WDI_wb","External_debt_stocks_wb",'IP_payments_wb','IP_receipts_wb','who_gvt_health_expenditure',"month_wage",
           'Total_reserves_wb','Number_of_new_businesses_registered_wb','Imports_wb','Exports_wb','GDP_pc_wb','GDP_deflator_wb','Govt_exp_gdp_wb']

ohter_info = pd.read_csv(UNILATERAL_PANEL,skiprows=1,sep="\t",usecols=usecols)

#Fill forward to get old data (in case of missing)
for i in ["GDP_int","POP_int","Physicians_per_1000_wb","Nurses_per_1000_wb"]:
    ohter_info[i] = ohter_info.groupby(['iso3'], sort=False)[i].apply(lambda x: x.ffill(limit=3).bfill(limit=3))

#Keep mean >2010
ohter_info = ohter_info.loc[ohter_info["year"]>last_year_mean].groupby("iso3").mean().reset_index()

#Merge with region
ohter_info = pd.merge(ohter_info,region,how="outer")

#Raw numbers
ohter_info["Physicians"] = ohter_info["Physicians_per_1000_wb"]*ohter_info["POP_int"]/1000
ohter_info["Nurses"] = ohter_info["Physicians_per_1000_wb"]*ohter_info["POP_int"]/1000

#GDP per capita
ohter_info["gdppc"] = ohter_info["GDP_int"]/ohter_info["POP_int"]

#Convert to USD
for i in ['who_gvt_health_expenditure',"resource_revenue","resource_taxes","total_taxes_revenue","cit_revenue","Health_exp_gdp_wb","Govt_exp_educ_gdp_wb","Govt_exp_gdp_wb"]:
    ohter_info[i] = ohter_info[i]*ohter_info["GDP_int"]
    
ohter_info["resource_revenue_gdp"] = ohter_info["resource_revenue"]/ohter_info["GDP_int"]

ohter_info["Govt_exp_educ_gdp_wb"] /= 100
ohter_info["Health_exp_gdp_wb"] /= 100

remove = []
for i,x,y in zip(ohter_info["iso3"],ohter_info["GDP_int"]/ohter_info["POP_int"],ohter_info["month_wage"]):
    if (np.log(x/12/y)>np.log2(1.5)) or (np.log(x/12/y)<-2):
        print(i,x/12,y,np.abs(np.log(x/12/y)))
        plt.annotate(i,(x,y))
        remove.append(i)
plt.plot(ohter_info["GDP_int"]/ohter_info["POP_int"],ohter_info["month_wage"],".")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("GDP per capita")
plt.ylabel("Wage")

#Delete the problematic ones
ohter_info.loc[ohter_info["iso3"].isin(remove),"month_wage"] = np.nan


mod = smf.ols(formula='np.log(month_wage) ~ np.log(GDP_int) + np.log(POP_int)', data=ohter_info)

res = mod.fit()

print(res.summary())

ohter_info["pred_month_wage"] = np.exp(res.predict(ohter_info))
ohter_info.loc[np.isnan(ohter_info["month_wage"]),"month_wage"] = ohter_info.loc[np.isnan(ohter_info["month_wage"]),"pred_month_wage"]

del ohter_info["pred_month_wage"]
#Fix the departing ones
ohter_info.loc[ohter_info["iso3"]=="USA","month_wage"] =  4056 #https://www.bls.gov/news.release/pdf/wkyeng.pdf
ohter_info.loc[ohter_info["iso3"]=="AUS","month_wage"] =  5661 #https://www.abs.gov.au/statistics/labour/earnings-and-work-hours/average-weekly-earnings-australia/latest-release

#ADD CIT
ohter_info["cit"] = ohter_info["iso3"].map(iso3_to_cit)
ohter_info = ohter_info.drop_duplicates()

ohter_info.to_csv(info_expenditures_output, sep="\t", index=None)
 


ohter_info.head(2)

In [ ]:
iso3_to_gdp = ohter_info.set_index("iso3")["GDP_int"].dropna().to_dict()
pickle.dump(iso3_to_gdp,open(iso3_to_gdp_output,"wb+"))


In [ ]:
a = pd.read_csv(UNILATERAL_PANEL,skiprows=1,sep="\t",nrows=1).columns
[_ for _ in a if "govt_ex" in _.lower()]

In [ ]:
#Other info
df_region = ohter_info.drop(columns=["year"])
df_region = df_region.rename(columns={"iso3":"iso3_d"})
df_region = df_region.drop_duplicates(subset=["iso3_d"]).dropna(subset=["iso3_d"])
iso3_to_region = df_region.set_index("iso3_d")["region_tjn"]

In [ ]:
iso3_to_wages = ohter_info.set_index("iso3")["month_wage"].dropna().to_dict()
pickle.dump(iso3_to_wages,open(iso3_to_wages_output,"wb+"))


In [ ]:
ohter_info.set_index("iso3")["month_wage"]

## 2.3 Gravity dataset

In [ ]:
def add_eq(df_dist,list_c,eq):
    """copies the data of another country"""
    for c in list_c:
        x = df_dist.loc[df_dist["iso3_o"]==eq]
        x["iso3_o"] = c
        df_dist = pd.concat([df_dist,x])
        x = df_dist.loc[df_dist["iso3_d"]==eq]
        x["iso3_d"] = c
        df_dist = pd.concat([df_dist,x])
    return df_dist



In [ ]:
# Read gravity variables
df_dist = pd.read_stata(GRAV_FILE)
df_dist = df_dist.loc[df_dist["year"]>last_year_mean].groupby(["iso3_o","iso3_d"]).mean().reset_index()

df_dist["iso3_o"] = df_dist["iso3_o"].str.replace("ROM","ROU")
df_dist["iso3_d"] = df_dist["iso3_d"].str.replace("ROM","ROU")
df_dist["iso3_o"] = df_dist["iso3_o"].str.replace("TLS","TMP")
df_dist["iso3_d"] = df_dist["iso3_d"].str.replace("TLS","TMP")
# df_dist["iso3_o"] = df_dist["iso3_o"].str.replace("ANT","CUW")
# df_dist["iso3_d"] = df_dist["iso3_d"].str.replace("ANT","CUW")
df_dist = df_dist.loc[(df_dist["iso3_o"]!="ANT")&(df_dist["iso3_d"]!="ANT")]

df_dist = add_eq(df_dist,["JEY","IMN","GGY"],"GBR")
# df_dist = add_eq(df_dist,["SRB","MNE"],"ALB")
# df_dist = add_eq(df_dist,["MCO"],"AND")
# df_dist = add_eq(df_dist,["SSD"],"SDN")
# df_dist = add_eq(df_dist,["COD"],"COG")
df_dist = add_eq(df_dist,["TLS"],"COG")


In [ ]:
df_dist["iso3_d"].value_counts().head(5) #make sure there are no duplicates

In [ ]:
set(df_fin["iso3_d"]) - set(df_dist["iso3_o"])

In [ ]:
for i in set(df_fin["iso3_d"]) - set(df_dist["iso3_o"]):
    try:
        iso3_to_name(i)
    except:
        print(i)

## 2.4 Variables from Linkedin

In [ ]:
# Linkedin variables
df_link = pd.read_csv(LINK_FILE,sep="\t",index_col=0,na_values=[""],keep_default_na=False)
df_link["iso3_d"] = df_link["index"].map(get_iso3)
df_link = df_link.drop(columns=["index","country","ln_pi"])
df_link = df_link.drop_duplicates(subset=["iso3_d"])

## 2.5 Bilateral info

In [ ]:
df_bil = pd.read_csv(BILATERAL_CROSS,skiprows=1,sep="\t")
df_bil = df_bil.groupby(["r_iso3","p_iso3"]).mean().reset_index()
df_bil = df_bil.loc[:,["r_iso3","p_iso3", 'Export', 'Import','dClaims','dLiabilities','PortI_inward','PortI_outward', 'FDI_inward', 'FDI_outward']]
for v in ['Export', 'Import','dClaims','dLiabilities','PortI_inward','PortI_outward', 'FDI_inward', 'FDI_outward']:
    df_bil["ln_"+v] = np.log(df_bil[v]+1)
    del df_bil[v]
    
df_bil = df_bil.rename(columns={"r_iso3":"iso3_o", "p_iso3": "iso3_d"})

df_bil

## 2.5 Combine datasets into a bilateral dataset

In [ ]:
iso3_to_etr = df_etrs["ETR_total"].to_dict()

In [ ]:
## Dataset to impute (financials and weird stuff) (destination files)
# Keep the non-bilateral columns (note the drop_duplicates)
x = df_dist.loc[:,[_ for _ in df_dist.columns if  (_[-2:]=="_d") and (_ not in ["heg_d","gsp_d_d","gsp_o_d","eu_d","tradeflow_comtrade_d","tradeflow_imf_d"])]].drop_duplicates()
print(x.shape) #should be around 257 countries

df_destination = pd.merge(x,df_region,how="outer") 
df_destination = pd.merge(df_destination,df_link.drop(columns=["cit","ln_gdp"]),how="left",on=["iso3_d"],suffixes=["","_d"])
df_destination["n_companies_orb"] = df_destination["iso3_d"].map(expected_companies).fillna(0) 

# Add efective tax rate
df_destination["etr_real_d"] = df_destination["iso3_d"].map(iso3_to_etr)
df_destination["ln_etr_real_d"] = np.log(0.001+df_destination["etr_real_d"])
df_destination = df_destination.dropna(subset=["GDP_int"])

remove_cols_dest = df_destination.columns

# Take the logarithm for many variiables
columns = np.array(df_destination.columns)[df_destination.dtypes!=object]
for var in columns: 
    if (var == "english") or (var == "etr") or ("ln" in var) or ("comrelig" in var) or ("comleg" in var) or ("cit" == var): #TO REVIEW if last clause if correct - Introduced to avoid error running the notebook
        continue
    if len(df_destination.loc[:,var].unique()) < 5:
        continue
    m = np.min(df_destination.loc[:,var].dropna())
    if m>0:
        m = 0
#     print(var,m,np.mean(df_destination.loc[:,var]-m)/(np.median(df_destination.loc[:,var].dropna()-m)))
    if ("GDP" in var) or (var == "distw") or ((np.mean(df_destination.loc[:,var]-m) > 2.*np.median(df_destination.loc[:,var].dropna()-m))):# or (var in ["Total FSI","cthi"]):
        print(var,m,np.mean(df_destination.loc[:,var]-m)/(np.median(df_destination.loc[:,var].dropna()-m)))
        df_destination["ln_"+var] = np.log(1+df_destination.loc[:,var]-m)
        del df_destination[var]
        
        
        

In [ ]:
df_destination = df_destination.loc[(df_destination["iso3_d"]!="WLD")] 

columns = np.array(df_destination.columns)[df_destination.dtypes!=object]
# df_destination = impute(df_destination,columns)


df_destination.columns = [_+"_d" if _[-2:]!="_d" else _ for _ in df_destination.columns]

for var in ["ln_resource_revenue_d", "ln_resource_revenue_gdp_d","ln_total_taxes_revenue_d","ln_cit_revenue_d"]:
    df_destination.loc[df_destination[var]<0,var] = 0


df_destination2 = df_destination.copy()
df_destination2.columns = [_[:-1]+"o" if _[-2:]=="_d" else _ for _ in df_destination.columns] 

# #Add dummies
# df_destination = pd.concat([df_destination,pd.get_dummies(df_destination["iso3_d"])],axis=1)

print(df_destination.shape)

countries_include - set(df_destination["iso3_d"])

In [ ]:
#Create a full dataset
pairs = list(product(countries_include,repeat=2))
pairs = pd.DataFrame(pairs,columns=["iso3_o","iso3_d"])

pairs = pd.merge(pairs, df_fin,how="left",validate="1:1")
pairs = pd.merge(pairs,df_destination,how="left",on=["iso3_d"],validate="m:1")
pairs = pd.merge(pairs,df_destination2,how="left",suffixes=["_d","_o"],on=["iso3_o"],validate="m:1") 

df_dist2 = df_dist.loc[:,["iso3_o","iso3_d"]+[_ for _ in df_dist if (_ not in df_destination.columns) and (_ not in df_destination2.columns) and ("ln"+_ not in df_destination.columns) and ("ln"+_ not in df_destination2.columns) and (_ not in remove_cols_dest)]]
pairs = pd.merge(pairs,df_dist2,how="left",on=["iso3_o","iso3_d"],validate="1:1") 
pairs = pd.merge(pairs,df_bil,how="left",on=["iso3_o","iso3_d"],validate="1:1")

# pairs = pairs.drop(columns=["conflict","indepdate","sever","sib_conflict","pta_bb","fta_bb","fta_hmr"])
pairs = pairs.drop(columns=["sib_conflict"])


print(pairs.dropna(subset=["pi"]).shape)

In [ ]:
# Lines commented to hot fix issue - Originally not commented
# pairs["ln_etr_real_o"] = np.log(0.01+pairs["etr_real_o"])
# pairs["ln_etr_real_d"] = np.log(0.01+pairs["etr_real_d"])

In [ ]:
# Originally not existing
pairs["etr_real_o"] = np.exp(pairs["ln_etr_real_o"])-0.01

In [ ]:
pairs["ln_cit_o"] = np.log(0.01+pairs["cit_o"])
pairs["ln_cit_d"] = np.log(0.01+pairs["cit_d"])

In [ ]:
set(["gdp_o","pop_o","gdpcap_o","entry_cost_o","area_o","eu_o","eu_d","year","region_d","region_o"])-set(pairs.columns)

In [ ]:
pairs = pairs.drop(columns=["gdp_o","pop_o","gdpcap_o","entry_cost_o","eu_o","eu_d","year","region_d","region_o"])

In [ ]:
for i in pairs.columns:
    if len(pairs)-pairs[i].count() > 0 :
        print(i,len(pairs)-pairs[i].count())

In [ ]:
# #fill missing (very few values coming from the distancefile only)
# columns = np.array(pairs.columns)[pairs.dtypes!=object]
# columns = [_ for _ in columns if _ not in df_fin.columns]

# # pairs = impute(pairs,columns)

In [ ]:
for var in ["emp","revt","pi","t_at","distw"]:
    pairs["ln_"+var] = np.log(1+pairs.loc[:,var]-m)
    del pairs[var]

In [ ]:
pairs.to_csv(df_pairs_output,sep="\t",index=None)
df_fin.to_csv(df_fin_output,sep="\t",index=None)

In [ ]:
df_fin.head()

In [ ]:
pairs.head()

In [ ]:
df_region_notna = df_region.dropna(subset=["iso3_d","region_tjn"])
OAF = AFRIC = set(df_region_notna.loc[df_region_notna["region_tjn"]=="Africa","iso3_d"].dropna())
OAS = ASIAT = set(df_region_notna.loc[df_region_notna["region_tjn"]=="Asia","iso3_d"].dropna())
OTE = EUROP = set(df_region_notna.loc[df_region_notna["region_tjn"]=="Europe","iso3_d"].dropna())
OAM = LAC_sca = set(df_region_notna.loc[df_region_notna["region_tjn"].str.contains("America"),"iso3_d"].dropna())
GRPS = set(df_region_notna["iso3_d"].dropna())
LAC = set(df_region_notna.loc[df_region_notna["region_tjn"]=="Latin America and the Caribbean","iso3_d"].dropna())

d_groups2countries = {"OAF": OAF,
                     "AFRIC": AFRIC,
                     "OAS": OAS,
                     "ASIAT": ASIAT,
                     "OTE": OTE,
                     "EUROP": EUROP,
                     "OAM": OAM,
                     "LAC_sca": LAC_sca,
                     "GRPS": GRPS,
                     "LAC": LAC}

pickle.dump(d_groups2countries,open(d_groups2countries_output,"wb+"))
